# Typhoon Thai Dialect Isan — สำรวจ metadata

## เริ่มอ่านหัวข้อ 1–3 ก่อน: มีกี่คอลัมน์ ขาดอะไร และครอบคลุมใครบ้าง?

Notebook นี้สำรวจ **metadata ทุกแถว** ของชุดเสียงอีสาน ไม่ได้ดาวน์โหลดหรือถอดรหัสเสียง
- ครั้งแรกอ่าน schema/footer และคอลัมน์ metadata จาก Parquet ต้นทางผ่าน HTTP byte ranges
- ไม่อ่านคอลัมน์ audio และไม่บันทึกทั้ง Parquet shards / WAV
- เก็บ snapshot ของ metadata อย่างเดียวใน data/raw/typhoon_thai_dialect_isan/metadata/<revision>/
- ครั้งต่อไปใช้ snapshot ที่ตรวจ checksum แล้ว จึงรัน offline ได้
- ไม่ฝึก ไม่เลือก threshold ไม่เปลี่ยน split และไม่ทำนายอายุ/เพศจากเสียง

[Dataset card](https://huggingface.co/datasets/typhoon-ai/thai-dialect-isan-dataset) อธิบายว่าเป็นเสียงตอบคำถามภาษาอีสานสำหรับ ASR/dialect study
ชุดนี้ **ไม่มี label bonafide/spoof แบบ SEA-Spoof**; หากใช้เป็นชุดเสียงจริงเสริมต้องประกาศที่มาและเกณฑ์ยืนยันเสียงจริง
ข้อมูลจังหวัด/อำเภอหมายถึง **ที่อาศัย** ไม่ใช่จังหวัดเกิดหรือสำเนียงที่ตรวจฟังแล้วทุกคลิป

คำเตือนเรื่องสิทธิ์: header ของ dataset ระบุ Apache-2.0 แต่เนื้อหา card ระบุ CC-BY-4.0
ต้องยืนยันเงื่อนไขกับผู้สร้างก่อนนำเสียงไปใช้/เผยแพร่ ผลรายแถวและ demographic metadata ใน notebook นี้เก็บเฉพาะเครื่องและถูก gitignore


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import itertools
import json
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file() and (p / "data/exploration").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project.")
EDA_DIR = ROOT / "data/exploration/typhoon_thai_dialect_isan"
OUTPUT = EDA_DIR / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from typhoon_eda_utils import (DATASET, DEFAULT_REVISION, fetch_metadata,
                              parse_age, age_band, parse_filename, file_sha256)

REVISION = DEFAULT_REVISION  # pinned to the metadata audit revision, not moving 'main'
WORKERS = 4
SHOW_RAW_EXAMPLES = False
AGE_ORDER = ["<18", "18-29", "30-44", "45-59", "60+", "unknown"]
STARTED_AT = datetime.now(timezone.utc).isoformat()
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110})
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 25)
TABLES = {}
def show_table(name, table, limit=40):
    TABLES[name] = table.copy()
    display(table.head(limit))
    if len(table) > limit:
        print(f"Showing {limit}/{len(table)} rows; full table exported locally.")
def save_plot(name):
    plt.tight_layout()
    plt.savefig(OUTPUT / (name + ".png"), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
def strings(frame, column):
    return frame[column].astype("string").fillna("").str.strip()
def counts(frame, columns):
    return frame.groupby(columns, dropna=False, observed=True).size().rename("clips").reset_index()
def normalized_text(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(value))).strip()

metadata_path, provenance = fetch_metadata(ROOT, REVISION, workers=WORKERS)
df = pq.read_table(metadata_path).to_pandas()
source_columns = [item["column"] for item in provenance["source_columns"]]
metadata_columns = [c for c in source_columns if c != "audio"]
assert "audio" not in df.columns and len(df) == provenance["rows"]
versions = {name: importlib.metadata.version(name) for name in
            ["pandas", "pyarrow", "huggingface_hub", "matplotlib", "seaborn", "nbformat"]}
print(f"Metadata loaded: {len(df):,} rows. Audio downloaded: 0 files.")


## 1. สรุปง่าย: มีกี่คอลัมน์และข้อมูลขาดตรงไหน?

แยกคอลัมน์ต้นทาง / คอลัมน์ที่เราอ่าน / คอลัมน์ที่เพิ่มเพื่อตรวจสอบ ไม่เรียกทุกคอลัมน์ว่า input ของ AASIST/RawNet2
Missing = null หรือข้อความว่างหลัง trim; gender=x เป็น **ไม่ระบุ** ตาม card ไม่ใช่ชาย/หญิงและไม่ถือว่าเป็นค่าว่าง
ค่าว่างและความขัดแย้งของข้อมูลเป็นคนละปัญหา ต้องตรวจทั้งสองอย่าง
ตัวเลข duration/ชั่วโมงมาจาก metadata ยังไม่ได้วัด header ของไฟล์เสียง


In [ ]:
profile_rows = []
for column in metadata_columns:
    s = strings(df, column)
    profile_rows.append({"column": column, "dtype": str(df[column].dtype),
        "missing_rows": int(s.eq("").sum()), "missing_pct": float(s.eq("").mean() * 100),
        "unique_nonblank_values": int(s.loc[s.ne("")].nunique())})
profile = pd.DataFrame(profile_rows).sort_values(["missing_rows", "column"], ascending=[False, True])
df["age_numeric"] = pd.to_numeric(df.age.map(parse_age), errors="coerce")
df["duration_seconds"] = pd.to_numeric(df.duration, errors="coerce")
df["gender_code"] = strings(df, "gender").str.lower().replace("", "(missing)")
df["province_value"] = strings(df, "province").replace("", "(missing)")
df["district_value"] = strings(df, "district").replace("", "(missing)")
parsed_names = pd.DataFrame(df.name.map(parse_filename).tolist(), index=df.index)
df = pd.concat([df, parsed_names], axis=1)
df["age_band"] = df.age_numeric.map(age_band)
valid_duration = df.duration_seconds.notna() & np.isfinite(df.duration_seconds) & df.duration_seconds.gt(0)
metadata_hours = float(df.loc[valid_duration, "duration_seconds"].sum() / 3600)
valid_age = df.age_numeric.notna()
basic_summary = pd.DataFrame([
 {"item": "Source columns including audio", "value": len(source_columns)},
 {"item": "Source metadata columns read", "value": len(metadata_columns)},
 {"item": "Added provenance columns (official_split/source_parquet)", "value": 2},
 {"item": "Total clips", "value": len(df)},
 {"item": "Metadata hours (not header-verified)", "value": round(metadata_hours, 3)},
 {"item": "Candidate speaker keys (not confirmed people)", "value": df.loc[df.candidate_speaker_key.ne(""), "candidate_speaker_key"].nunique()},
 {"item": "Downloaded/decoded audio files", "value": 0},
])
show_table("quick_summary", basic_summary)
show_table("missing_data", profile)
show_table("missing_data_nonzero", profile.loc[profile.missing_rows.gt(0)])
demographic_status = pd.DataFrame([
 {"dimension": "age", "blank_rows": int(strings(df, "age").eq("").sum()),
  "usable_or_known_rows": int(valid_age.sum()), "additional_unspecified_or_invalid": int((~valid_age & strings(df, "age").ne("")).sum())},
 {"dimension": "gender", "blank_rows": int(strings(df, "gender").eq("").sum()),
  "usable_or_known_rows": int(df.gender_code.isin(["m", "f"]).sum()), "additional_unspecified_or_invalid": int(df.gender_code.eq("x").sum())},
 {"dimension": "province", "blank_rows": int(strings(df, "province").eq("").sum()),
  "usable_or_known_rows": int(strings(df, "province").ne("").sum()), "additional_unspecified_or_invalid": 0},
 {"dimension": "district", "blank_rows": int(strings(df, "district").eq("").sum()),
  "usable_or_known_rows": int(strings(df, "district").ne("").sum()), "additional_unspecified_or_invalid": 0},
])
show_table("demographic_completeness", demographic_status)
display(Markdown(f"**อ่านง่าย:** มี {len(source_columns)} คอลัมน์ต้นทาง รวม audio; เราอ่าน {len(metadata_columns)} "
    f"คอลัมน์ metadata จาก {len(df):,} คลิป รวม {metadata_hours:.2f} ชั่วโมงตาม metadata. "
    f"ยังไม่มี waveform/header/SNR/sample-rate ที่เราตรวจเอง."))


## 2. Data dictionary และข้อที่ต้องระวัง

province/district = ที่อาศัยตาม card
age ใน card อธิบายเป็นอายุ แต่ dtype ที่ตรวจจริงอาจเป็น string เช่น "24.0"; แสดง schema จริงควบคู่
third component ของ name ใช้เป็น **candidate speaker key** เท่านั้น ห้ามถือว่า 1 รหัส = 1 บุคคลแน่นอนก่อนผู้สร้างยืนยัน
อักษรนำหน้ารหัสไม่ใช้เติม gender ที่หาย; gen/fin เป็น domain code ใน filename ไม่ใช่ label จริง/ปลอม


In [ ]:
notes = {
 "id": "รหัสคลิปตามต้นทาง; ตรวจ unique",
 "audio": "คอลัมน์เสียงในต้นทาง — ไม่ได้อ่าน/ดาวน์โหลด/ถอดรหัสใน EDA นี้",
 "raw": "transcript มี annotation [อีสาน]<ไทยมาตรฐาน>; ไม่ใช่ waveform",
 "thai_spelling": "transcript ที่ใช้การสะกดแบบไทยมาตรฐาน; ไม่เปลี่ยนภาษาของเสียงจริง",
 "isan_spelling": "transcript สะกดแบบอีสาน",
 "name": "filename มีรหัส; ส่วนที่สามเป็น candidate speaker key ที่ต้องยืนยัน",
 "district": "อำเภอที่ผู้พูดอาศัย ตาม dataset card",
 "province": "จังหวัดที่ผู้พูดอาศัย; ไม่ใช่จังหวัดเกิดหรือสำเนียงยืนยัน",
 "age": "อายุตาม metadata; parse โดยไม่เติมค่าที่หาย",
 "gender": "m/f ตาม metadata; x=not specified; blank แยกจาก x",
 "question_id": "รหัสคำถาม prompt; ซ้ำได้เมื่อหลายคนตอบคำถามเดียวกัน",
 "question": "ข้อความคำถามที่ใช้กระตุ้นการพูด ไม่ใช่สิ่งที่ผู้พูดตอบ",
 "duration": "ความยาววินาทีตาม metadata ยังไม่ยืนยันกับไฟล์เสียงจริง",
}
dictionary = pd.DataFrame(provenance["source_columns"])
dictionary["meaning"] = dictionary.column.map(notes).fillna("ฟิลด์ต้นทางเพิ่มเติม: ต้องสอบทานความหมาย")
show_table("source_schema_dictionary", dictionary)
shard_inventory = pd.DataFrame([{"file": r["file"], "split": r["split"], "rows": r["rows"]}
                               for r in provenance["files"]])
show_table("metadata_shard_inventory", shard_inventory)
print("Pinned revision:", provenance["revision"])
print("Snapshot checksum:", provenance["metadata_sha256"])
print("Audio column read:", provenance["audio_column_read"])
if SHOW_RAW_EXAMPLES:
    display(df[metadata_columns + ["official_split"]].head(5))
else:
    print("Raw transcript/ID examples hidden; outputs stay local.")
schema_variants = {json.dumps(r["source_schema"], sort_keys=True) for r in provenance["files"]}
print("Distinct source schema variants:", len(schema_variants))


## 3. ภาพรวม split และคุณภาพ metadata

Train/Test เป็น split ของ **งาน ASR ต้นทาง** ไม่ใช่ Dev/Test ของ SEA-Spoof
ยังไม่ยืนยันการแยกผู้พูด ต้องตรวจ candidate key overlap ภายหลัง
อายุที่แปลงไม่ได้ยังเป็น unknown ไม่ทำนายแทน; อายุไม่ใช่จำนวนเต็มแสดง warning แยก
ID/text/prompt ซ้ำมีความหมายต่างกัน ไม่เปลี่ยน split หรือทิ้งแถวอัตโนมัติ


In [ ]:
show_table("split_summary", df.groupby("official_split").agg(
    clips=("id", "size"), unique_ids=("id", "nunique"),
    metadata_seconds=("duration_seconds", "sum"),
    candidate_keys=("candidate_speaker_key", lambda x: x.loc[x.ne("")].nunique())).reset_index())
integrity = pd.DataFrame([
 {"check": "blank id", "rows": int(strings(df, "id").eq("").sum())},
 {"check": "duplicate id (all occurrences)", "rows": int(df.id.duplicated(keep=False).sum())},
 {"check": "duplicate filename (all occurrences)", "rows": int(df.name.duplicated(keep=False).sum())},
 {"check": "unparsed candidate speaker key", "rows": int(df.filename_parse_status.ne("parsed").sum())},
 {"check": "invalid/missing duration", "rows": int((~valid_duration).sum())},
 {"check": "invalid nonblank age", "rows": int((~valid_age & strings(df, "age").ne("")).sum())},
 {"check": "fractional age values", "rows": int((valid_age & df.age_numeric.mod(1).ne(0)).sum())},
 {"check": "unexpected nonblank gender code", "rows": int((~df.gender_code.isin(["m", "f", "x", "(missing)"])).sum())},
])
show_table("integrity_checks", integrity)
missing_by_split = pd.DataFrame({split: pd.Series({
    c: float(strings(group, c).eq("").mean() * 100) for c in metadata_columns})
    for split, group in df.groupby("official_split")}).rename_axis("column").reset_index()
show_table("missing_pct_by_split", missing_by_split)
plt.figure(figsize=(9, max(5, .4 * len(metadata_columns))))
sns.heatmap(missing_by_split.set_index("column"), annot=True, fmt=".2f",
            vmin=0, vmax=100, cmap="YlOrRd", yticklabels=True)
plt.title("Blank metadata (%) by source split")
save_plot("01_missing_data")


## 4. ตรวจ candidate speaker keys และ metadata ขัดแย้ง

นับคลิปและรหัสแยกกัน เพื่อไม่ให้คนที่พูดหลายคลิปมีน้ำหนักมากเกิน
แยก "อายุหลายค่าที่ทราบ" ออกจาก "บางคลิปไม่ทราบอายุ"
ถ้าค่า age/gender/province/district ขัดกันภายในรหัส ไม่เลือกค่าใดเอง
อาจเป็น recording หลายเวลา, metadata error หรือความหมายรหัสที่ไม่ใช่บุคคล — ต้องถามต้นทาง


In [ ]:
speaker_rows = []
for key, group in df.loc[df.candidate_speaker_key.ne("")].groupby("candidate_speaker_key"):
    ages = sorted(group.age_numeric.dropna().unique().tolist())
    genders = sorted(set(group.gender_code) - {"(missing)"})
    provinces = sorted(set(group.province_value) - {"(missing)"})
    districts = sorted(set(group.district_value) - {"(missing)"})
    speaker_rows.append({
        "candidate_speaker_key": key, "clips": len(group),
        "metadata_hours": float(group.loc[group.duration_seconds.gt(0), "duration_seconds"].sum()/3600),
        "source_split_count": group.official_split.nunique(),
        "age_values": ";".join(str(v) for v in ages), "known_age_count": len(ages),
        "age_conflict": len(ages) > 1, "age_missing_clips": int(group.age_numeric.isna().sum()),
        "age_band_consistent": "conflicting" if len(ages)>1 else age_band(ages[0] if ages else None),
        "gender_values": ";".join(genders), "gender_conflict": len(genders)>1,
        "gender_consistent": "conflicting" if len(genders)>1 else genders[0] if genders else "(missing)",
        "province_values": ";".join(provinces), "province_conflict": len(provinces)>1,
        "province_consistent": "conflicting" if len(provinces)>1 else provinces[0] if provinces else "(missing)",
        "district_conflict": len(districts)>1,
    })
speakers = pd.DataFrame(speaker_rows)
show_table("candidate_key_clip_distribution", speakers[["clips", "metadata_hours"]].describe(
    percentiles=[.05, .25, .5, .75, .95]).reset_index())
conflicts = pd.DataFrame([{"field": field,
    "conflicting_candidate_keys": int(speakers[field + "_conflict"].sum())}
    for field in ["age", "gender", "province", "district"]])
show_table("candidate_key_conflicts", conflicts)
TABLES["candidate_key_details"] = speakers
TABLES["age_conflict_details"] = speakers.loc[speakers.age_conflict]
conflict_keys = set(speakers.loc[speakers.age_conflict, "candidate_speaker_key"])
df["age_key_conflict"] = df.candidate_speaker_key.isin(conflict_keys)
show_table("age_conflict_coverage", df.groupby("official_split").age_key_conflict.agg(
    ["size", "sum"]).reset_index().rename(columns={"size": "clips", "sum": "clips_on_age_conflict_keys"}))
sns.histplot(data=speakers, x="clips", bins=30)
plt.title("Clips per candidate speaker key (not verified persons)"); plt.xlabel("Clips per key")
save_plot("02_clips_per_candidate_key")


## 5. อายุ: ทั้งระดับคลิปและ candidate key

ช่วงอายุใช้เพื่อสำรวจ ไม่ใช่เกณฑ์ยืนยันว่าครอบคลุมทุกช่วง
กราฟระดับคลิปใช้อายุที่แต่ละแถวระบุ รวมคลิปของรหัสที่ขัดแย้ง
กราฟระดับรหัสจัดรหัสที่มีอายุหลายค่าเป็น conflicting จึงไม่นับซ้ำหลายช่วง
กลุ่มผู้สูงอายุที่มีหลายคลิปจากรหัสเดียว ยังไม่ใช่หลักฐานแทนผู้สูงอายุทั่วไป


In [ ]:
age_counts = df.age_band.value_counts().reindex(AGE_ORDER, fill_value=0).rename_axis("age_band").reset_index(name="clips")
age_counts["pct_clips"] = age_counts.clips / len(df) * 100
show_table("age_band_clips", age_counts)
key_age_order = AGE_ORDER + ["conflicting"]
key_age_counts = speakers.age_band_consistent.value_counts().reindex(key_age_order, fill_value=0).rename_axis(
    "age_band").reset_index(name="candidate_keys")
show_table("age_band_candidate_keys", key_age_counts)
show_table("age_numeric_summary", df.groupby("official_split").age_numeric.describe(
    percentiles=[.05, .25, .5, .75, .95]).reset_index())
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(age_counts.age_band, age_counts.clips)
axes[0].set(title="Age bands: clip counts", ylabel="Clips")
axes[1].bar(key_age_counts.age_band, key_age_counts.candidate_keys)
axes[1].tick_params(axis="x", rotation=35)
axes[1].set(title="Age bands: candidate keys (conflicts separate)", ylabel="Candidate keys")
save_plot("03_age_coverage")
sns.histplot(data=df.loc[valid_age], x="age_numeric", bins=range(0, 121, 2), hue="official_split", element="step")
plt.xlim(max(0, df.age_numeric.min()-2), df.age_numeric.max()+2)
plt.title("Age from metadata (not inferred from audio)"); plt.xlabel("Age")
save_plot("04_age_histogram")


## 6. เพศ: เก็บ x และ missing แยก ไม่เติมจาก filename

m/f ใช้ตาม metadata; x=not specified ตาม card ไม่ใช่เพศที่เราทำนาย
การวิเคราะห์ครบ age × gender × province แบบ m/f ไม่ได้หมายความว่าควรทิ้ง x จากทั้งชุด
รายงาน known/unspecified/missing แยก และเลือก protocol ก่อนเห็นคะแนนโมเดล


In [ ]:
gender_counts = df.gender_code.value_counts().rename_axis("gender").reset_index(name="clips")
gender_counts["pct_clips"] = gender_counts.clips / len(df) * 100
show_table("gender_clips", gender_counts)
gender_keys = speakers.gender_consistent.value_counts().rename_axis("gender").reset_index(name="candidate_keys")
show_table("gender_candidate_keys", gender_keys)
show_table("gender_by_split", counts(df, ["official_split", "gender_code"]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(gender_counts.gender, gender_counts.clips)
axes[1].bar(gender_keys.gender, gender_keys.candidate_keys)
axes[0].set(title="Gender metadata: clips", ylabel="Clips")
axes[1].set(title="Gender metadata: candidate keys", ylabel="Candidate keys")
save_plot("05_gender_coverage")


## 7. จังหวัด/อำเภอ: ที่อาศัย ไม่ใช่ภูมิลำเนาหรือสำเนียง

แสดงชื่อไทยในตาราง ส่วนกราฟใช้รหัส P01... เพื่อให้ไม่ขึ้นกับฟอนต์ไทยของเครื่อง
จังหวัดเดียวมีจำนวนคลิปมาก ไม่ได้หมายถึงการกระจายประชากรสมดุล
จำนวน candidate key ที่แตะจังหวัดนั้นอาจซ้ำข้ามจังหวัด ถ้า metadata ของรหัสเดียวไม่สอดคล้อง


In [ ]:
province_counts = df.groupby("province_value").agg(clips=("id", "size"),
    metadata_seconds=("duration_seconds", "sum"),
    candidate_keys_observed=("candidate_speaker_key", lambda x: x.loc[x.ne("")].nunique())).reset_index()
province_counts = province_counts.sort_values("clips", ascending=False).reset_index(drop=True)
province_counts["province_code"] = [f"P{i+1:02d}" for i in range(len(province_counts))]
province_counts["pct_clips"] = province_counts.clips / len(df) * 100
province_counts["metadata_hours"] = province_counts.metadata_seconds / 3600
show_table("province_coverage", province_counts)
district_counts = df.groupby(["province_value", "district_value"]).agg(
    clips=("id", "size"), candidate_keys_observed=("candidate_speaker_key", lambda x: x.loc[x.ne("")].nunique())).reset_index().sort_values("clips", ascending=False)
show_table("district_coverage", district_counts, limit=20)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(province_counts.province_code, province_counts.clips)
axes[1].bar(province_counts.province_code, province_counts.candidate_keys_observed)
axes[0].set(title="Residence province: clips (see code lookup table)", ylabel="Clips")
axes[1].set(title="Residence province: observed candidate keys", ylabel="Candidate keys")
for ax in axes: ax.tick_params(axis="x", rotation=45)
save_plot("06_province_coverage")
province_map = dict(zip(province_counts.province_value, province_counts.province_code))
df["province_code"] = df.province_value.map(province_map)


## 8. จุดตัด age × gender × province และขอบเขตที่ใช้ได้

ตรวจว่ามีคอลัมน์อย่างเดียวไม่พอ ต้องดูว่ามีครบในแถวเดียวกันกี่แถว
แยก completeness ก่อน/หลังตัดรหัสที่อายุขัดแย้ง และ x/unknown ไม่ถูกเติมเป็น m/f
ตารางนี้เสนอ coverage เท่านั้น ไม่เลือกกลุ่มตามผลทำนาย และไม่สร้าง train/test ใหม่


In [ ]:
df["complete_age_mf_province"] = valid_age & df.gender_code.isin(["m", "f"]) & df.province_value.ne("(missing)")
df["complete_age_mf_province_no_age_conflict"] = df.complete_age_mf_province & ~df.age_key_conflict & df.candidate_speaker_key.ne("")
completeness = pd.DataFrame([
 {"criterion": "valid age", "clips": int(valid_age.sum())},
 {"criterion": "gender m/f", "clips": int(df.gender_code.isin(["m", "f"]).sum())},
 {"criterion": "province nonblank", "clips": int(df.province_value.ne("(missing)").sum())},
 {"criterion": "valid age + m/f + province", "clips": int(df.complete_age_mf_province.sum())},
 {"criterion": "above + parsed key + no age conflict", "clips": int(df.complete_age_mf_province_no_age_conflict.sum())},
])
completeness["pct_all_clips"] = completeness.clips / len(df) * 100
show_table("joint_completeness", completeness)
show_table("age_gender_coverage", counts(df, ["age_band", "gender_code"]))
joint = df.groupby(["age_band", "gender_code", "province_value"], observed=True).agg(
    clips=("id", "size"), candidate_keys_observed=("candidate_speaker_key", lambda x: x.loc[x.ne("")].nunique())).reset_index()
show_table("age_gender_province_coverage", joint, limit=20)
age_gender_heat = pd.crosstab(df.age_band, df.gender_code).reindex(AGE_ORDER, fill_value=0)
sns.heatmap(age_gender_heat, annot=True, fmt="d", cmap="Blues")
plt.title("Age x gender: clip counts (not independent persons)")
save_plot("07_age_gender")
show_table("consistent_key_age_gender", counts(
    speakers.rename(columns={"age_band_consistent":"age_band", "gender_consistent":"gender_code"}),
    ["age_band", "gender_code"]).rename(columns={"clips": "candidate_keys"}))


## 9. Duration ตาม metadata และ input window ของ detector

duration เป็นวินาทีจากต้นทาง ยังไม่ใช่การเปิดไฟล์ตรวจ frames/sample rate
รายงาน distribution และ IQR warnings เพื่อสอบทาน ไม่ตัด outliers อัตโนมัติ
เทียบ 4.0375 วินาทีเพื่อเห็น crop/repeat-pad โดยประมาณ ไม่ใช่การเปลี่ยนโมเดลหรือวัดผล


In [ ]:
duration_data = df.loc[valid_duration].copy()
show_table("duration_summary", duration_data.groupby("official_split").duration_seconds.describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]).reset_index())
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=duration_data, x="duration_seconds", hue="official_split", bins=50, element="step", ax=axes[0])
sns.ecdfplot(data=duration_data, x="duration_seconds", hue="official_split", ax=axes[1])
axes[0].set(title="Metadata duration histogram", xlabel="Seconds")
axes[1].set(title="Metadata duration ECDF", xlabel="Seconds")
save_plot("08_metadata_duration")
show_table("duration_by_age", duration_data.groupby("age_band").duration_seconds.agg(
    ["count", "median", "mean", "min", "max"]).reset_index())
WINDOW_S = 64_600 / 16_000
duration_data["estimated_window_action"] = np.where(
    duration_data.duration_seconds < WINDOW_S, "repeat_pad",
    np.where(duration_data.duration_seconds > WINDOW_S, "trim", "exact"))
show_table("estimated_input_window", counts(duration_data, ["official_split", "estimated_window_action"]))
duration_data["duration_iqr_warning"] = False
for _, group in duration_data.groupby("official_split"):
    q1, q3 = group.duration_seconds.quantile([.25, .75])
    duration_data.loc[group.index, "duration_iqr_warning"] = (
        group.duration_seconds.lt(q1 - 1.5*(q3-q1)) | group.duration_seconds.gt(q3 + 1.5*(q3-q1)))
TABLES["duration_iqr_warning_details"] = duration_data.loc[duration_data.duration_iqr_warning,
    ["id", "name", "official_split", "duration_seconds"]]
print("Duration-based estimates only; no audio headers decoded.")


## 10. Transcript, question และ prompt domain

raw มี markup อีสาน→ไทยมาตรฐาน; ไม่ควรเรียกข้อความสามแบบว่าเสียงสามชุด
นับ characters ไม่ใช่จำนวนคำไทย; ไม่เติม transcript ที่หายด้วยโมเดล
หลายคนตอบคำถามเดียวกันจึงมี question_id ซ้ำได้
ข้อความซ้ำหรือ question overlap ไม่ยืนยัน audio leakage แต่ช่วยอธิบาย protocol


In [ ]:
for column in ["raw", "thai_spelling", "isan_spelling", "question"]:
    df[column + "_normalized"] = df[column].map(normalized_text)
    df[column + "_chars"] = df[column + "_normalized"].str.len()
transcript_quality = pd.DataFrame([{
    "column": column, "blank_clips": int(df[column + "_chars"].eq(0).sum()),
    "unique_nonblank_normalized": df.loc[df[column + "_chars"].gt(0), column + "_normalized"].nunique(),
    "median_chars": float(df[column + "_chars"].median()),
    "max_chars": int(df[column + "_chars"].max())}
    for column in ["raw", "thai_spelling", "isan_spelling", "question"]])
show_table("transcript_quality", transcript_quality)
df["has_raw_annotation_pair"] = df.raw_normalized.map(lambda x: bool(re.search(r"\[[^\]]+\]<[^>]+>", x)))
show_table("annotation_coverage", counts(df, ["official_split", "has_raw_annotation_pair"]))
show_table("prompt_domain_counts", counts(df, ["official_split", "prompt_domain_code"]))
question_usage = df.groupby("question_id").agg(clips=("id", "size"),
    text_variants=("question_normalized", "nunique"),
    candidate_keys=("candidate_speaker_key", lambda x: x.loc[x.ne("")].nunique())).reset_index()
TABLES["question_usage_details"] = question_usage
show_table("question_usage_distribution", question_usage[["clips", "candidate_keys", "text_variants"]].describe().reset_index())
show_table("question_text_consistency", pd.DataFrame([
    {"check": "question IDs with >1 normalized text", "count": int(question_usage.text_variants.gt(1).sum())}]))
sns.histplot(data=df, x="isan_spelling_chars", hue="official_split", bins=50, element="step")
plt.title("Isan transcript character lengths (not word counts)")
save_plot("09_transcript_lengths")


## 11. Train/Test overlap: รหัสคลิป, candidate speaker, คำถาม และข้อความ

Train/Test ของ ASR อาจใช้ผู้พูดคนเดิมในการทดสอบ; ห้ามอ้างว่าเป็น speaker-disjoint ก่อนตรวจ
ไม่เรียก candidate key ที่ซ้ำว่าเป็นบุคคลยืนยัน แต่ถือเป็นประเด็นสอบทานที่ต้องรายงาน
หากใช้ทั้งชุดเป็น external genuine-only test ห้ามนำไฟล์ใดไปฝึก/ปรับ threshold ในรอบนั้น
ยังไม่มี audio hashes จึงไม่สรุปว่าไฟล์เสียงไม่ซ้ำ


In [ ]:
split_overlap_rows = []
for split_a, split_b in itertools.combinations(sorted(df.official_split.unique()), 2):
    a, b = df.loc[df.official_split.eq(split_a)], df.loc[df.official_split.eq(split_b)]
    for key in ["id", "name", "candidate_speaker_key", "question_id",
                "thai_spelling_normalized", "isan_spelling_normalized"]:
        values_a = set(strings(a, key)) - {""}
        values_b = set(strings(b, key)) - {""}
        common = values_a & values_b
        split_overlap_rows.append({"split_a": split_a, "split_b": split_b, "key": key,
            "shared_unique_values": len(common),
            "clips_in_a_with_shared_value": int(strings(a, key).isin(common).sum()),
            "clips_in_b_with_shared_value": int(strings(b, key).isin(common).sum())})
split_overlap = pd.DataFrame(split_overlap_rows)
show_table("split_overlap", split_overlap)
shared_candidate_keys = set(df.loc[df.official_split.eq("train"), "candidate_speaker_key"]) & set(
    df.loc[df.official_split.eq("test"), "candidate_speaker_key"]) - {""}
TABLES["overlapping_candidate_key_details"] = speakers.loc[speakers.candidate_speaker_key.isin(shared_candidate_keys)]
print(f"Candidate keys in both source splits: {len(shared_candidate_keys)} (not verified persons).")


## 12. สิ่งที่ตรวจแล้ว / ยังตรวจไม่ได้ และคำถามให้อาจารย์

ตรวจได้: schema, missing, demographics ตาม metadata, ความขัดแย้งภายในรหัส, duration metadata, split overlap
ยังตรวจไม่ได้: actual sample rate/channels, waveform, silence/clipping, audio hashes, SNR/คุณภาพฟัง, speaker identity, overlap กับ SEA-Spoof

สำหรับงานหลัก: SEA-Spoof มี label จริง/ปลอม; Typhoon เป็น speech corpus ไม่มี label anti-spoofing รายแถว
เสนอใช้ Typhoon เป็น **ชุดทดสอบเสียงจริงภายนอกเชิงสำรวจ** หลังยืนยันสิทธิ์/ที่มา/คุณภาพ
ใช้ threshold ที่เลือกจาก SEA-Spoof Dev เท่านั้น แล้วรายงานเสียงจริงถูกทายเป็น spoof (FRR) รายกลุ่ม
ชุดเสียงจริงล้วนไม่สามารถคำนวณ EER หรือความสามารถตรวจเสียงปลอมรายวัยได้
ไม่อ้างว่าจังหวัดที่อาศัยพิสูจน์สำเนียง และไม่อ้างครอบคลุมผู้สูงอายุ/ทุกภาคจากจำนวนคลิปเพียงอย่างเดียว


In [ ]:
scope_status = pd.DataFrame([
 {"item": "Source schema / metadata every row", "status": "checked"},
 {"item": "Age / gender / residence coverage", "status": "checked as published metadata"},
 {"item": "Candidate filename keys and split overlap", "status": "checked; identity unverified"},
 {"item": "Actual audio headers / waveform / file hashes", "status": "not checked; no audio downloaded"},
 {"item": "Age/gender inferred from voice", "status": "not performed"},
 {"item": "AASIST/RawNet2 inference or fine-tuning", "status": "not performed"},
 {"item": "Speaker overlap with SEA-Spoof", "status": "not verified"},
 {"item": "License for subsequent audio usage", "status": "needs confirmation: header Apache-2.0 vs card CC-BY-4.0"},
])
show_table("scope_status", scope_status)
legacy_path = ROOT / "results/dataset_metadata_audit.json"
legacy_comparison = []
if legacy_path.is_file():
    legacy = json.loads(legacy_path.read_text(encoding="utf-8")).get("typhoon", {})
    if legacy.get("revision") == REVISION:
        for metric, actual in [("clips", len(df)), ("candidate_speakers", len(speakers)),
                               ("age_known_clips", int(valid_age.sum()))]:
            legacy_comparison.append({"metric": metric, "previous_audit": legacy.get("combined", {}).get(metric),
                                      "this_run": actual})
        show_table("previous_audit_crosscheck", pd.DataFrame(legacy_comparison))

summary = {
 "dataset": DATASET, "revision": REVISION,
 "source_card": "https://huggingface.co/datasets/" + DATASET,
 "started_at_utc": STARTED_AT, "finished_at_utc": datetime.now(timezone.utc).isoformat(),
 "scope": {"metadata_only": True, "audio_files_downloaded": 0, "audio_headers_checked": False,
           "source_columns_including_audio": len(source_columns), "metadata_columns_read": len(metadata_columns),
           "clips": len(df), "metadata_hours": metadata_hours, "candidate_keys": len(speakers),
           "speaker_keys_verified_as_persons": False, "source_split_overlap_candidate_keys": len(shared_candidate_keys)},
 "metadata_snapshot": str(metadata_path.relative_to(ROOT)), "metadata_sha256": provenance["metadata_sha256"],
 "versions": versions, "python_executable": sys.executable,
 "missing": profile.to_dict(orient="records"),
 "age_range": [float(df.age_numeric.min()), float(df.age_numeric.max())],
 "age_conflicting_candidate_keys": int(speakers.age_conflict.sum()),
 "gender_counts": gender_counts.to_dict(orient="records"),
 "joint_completeness": completeness.to_dict(orient="records"),
 "limitations": ["Metadata durations are not verified against audio headers.",
    "No anti-spoofing label supplied by source; no synthetic group for EER.",
    "Filename candidate keys are not verified identities.",
    "Residence province is not verified birthplace/accent.",
    "x gender retained as unspecified; missing and conflicts are reported separately.",
    "No waveform, audio duplicate hashes, model predictions or SNR analysis.",
    "Data is demographically imbalanced; do not claim all ages/regions coverage.",
    "License inconsistency must be resolved before subsequent audio usage."]}
TABLES["metadata_row_quality"] = df.drop(columns=[c for c in df.columns if c.startswith(
    ("raw", "thai_spelling", "isan_spelling", "question"))], errors="ignore")
for name, table in TABLES.items():
    table.to_csv(OUTPUT / (name + ".csv"), index=False, encoding="utf-8-sig")
(OUTPUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
display(Markdown(
    f"### สรุปสำหรับคุยอาจารย์\n"
    f"- ต้นทาง **{len(source_columns)} คอลัมน์** รวม audio; อ่าน metadata **{len(metadata_columns)} คอลัมน์**\n"
    f"- **{len(df):,} คลิป / {metadata_hours:.2f} ชั่วโมง** ตาม metadata; ยังไม่ได้โหลดเสียง\n"
    f"- อายุที่ระบุ **{df.age_numeric.min():.0f}–{df.age_numeric.max():.0f} ปี**; age conflict **{int(speakers.age_conflict.sum())} รหัส**\n"
    f"- รหัสผู้พูดจาก filename **{len(speakers)} รหัส** ยังไม่ยืนยันว่าเป็นจำนวนคน\n"
    f"- candidate keys ข้าม Train/Test **{len(shared_candidate_keys)} รหัส**; ไม่อ้าง speaker-disjoint\n"
    f"- อายุ 18–29 มี **{df.age_band.eq('18-29').mean()*100:.2f}%** ของคลิป จึงไม่สมดุลทุกวัย\n"
    f"- ผลครบใน data/exploration/typhoon_thai_dialect_isan/outputs/; ไม่ใช้ข้อมูลนี้เลือก threshold"
))
print("Metadata EDA complete. Audio was not read, saved or decoded.")
